In [1]:
import math

In [2]:
# ---------------------------------------------------------
# 1. OUR TINY DATABASE (THE LIBRARY)
# ---------------------------------------------------------
documents = {
    "doc1": "The ultimate luxury hotel and resort resort resort for felines.", 
    "doc2": "A technical manual for router hardware model-X configurations.",
    "doc3": "This short poem talks about a wild giraffe in its natural safari habitat."
}

In [3]:
# ---------------------------------------------------------
# 2. ASSISTANT #1: THE BM25 ROBOT (From First Principles)
# ---------------------------------------------------------
class BM25Retriever:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = {k: v.lower().replace(".", "").split() for k, v in docs.items()}
        self.k1 = k1  # Block 2: Saturation constant
        self.b = b    # Block 3: Length penalty constant
        
        # Calculate background library stats
        self.total_docs = len(self.docs)
        self.avg_doc_len = sum(len(d) for d in self.docs.values()) / self.total_docs
        
        # Build an Inverted Index (Vocabulary count across all documents)
        self.doc_frequencies = {}
        for doc_words in self.docs.values():
            unique_words = set(doc_words)
            for word in unique_words:
                self.doc_frequencies[word] = self.doc_frequencies.get(word, 0) + 1

    def _get_idf(self, word):
        """BLOCK 1: WORD RARITY (Inverse Document Frequency)"""
        df = self.doc_frequencies.get(word, 0)
        if df == 0: return 0
        # Standard BM25 IDF formula
        return math.log(1 + (self.total_docs - df + 0.5) / (df + 0.5))

    def score_document(self, query_words, doc_id):
        doc_words = self.docs[doc_id]
        doc_len = len(doc_words)
        score = 0.0
        
        for word in query_words:
            if word not in doc_words:
                continue
                
            # Block 1: Get word rarity
            idf = self._get_idf(word)
            
            # Block 2: Get term frequency
            tf = doc_words.count(word)
            
            # Block 3: Length penalty calculation
            length_normalization = 1.0 - self.b + (self.b * (doc_len / self.avg_doc_len))
            
            # Combine Block 2 & Block 3 into saturated TF
            saturated_tf = (tf * (self.k1 + 1)) / (tf + self.k1 * length_normalization)
            
            # Multiply everything together and accumulate
            score += idf * saturated_tf
            
        return score

    def retrieve(self, query):
        query_words = query.lower().split()
        scores = {doc_id: self.score_document(query_words, doc_id) for doc_id in self.docs}
        # Sort documents by highest score
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)


In [4]:
# ---------------------------------------------------------
# 3. ASSISTANT #2: THE VECTOR READER (Simulated Embedding Space)
# ---------------------------------------------------------
class SimulatedVectorRetriever:
    """
    Real vector search uses complex neural networks (like OpenAI or BERT).
    To keep this code dependency-free, we simulate semantic understanding 
    using a simple 'concept map' lookup table.
    """
    def __init__(self):
        # We manually map semantic concepts to our documents
        self.semantic_database = {
            "doc1": {"cat_vibe": 0.95, "hotel_vibe": 0.90, "router_vibe": 0.00, "animal_vibe": 0.40},
            "doc2": {"cat_vibe": 0.00, "hotel_vibe": 0.10, "router_vibe": 0.98, "animal_vibe": 0.00},
            "doc3": {"cat_vibe": 0.10, "hotel_vibe": 0.00, "router_vibe": 0.00, "animal_vibe": 0.92}
        }

    def retrieve(self, query):
        query = query.lower()
        # Parse query string to guess user intent / vibes
        query_vector = {"cat_vibe": 0.0, "hotel_vibe": 0.0, "router_vibe": 0.0, "animal_vibe": 0.0}
        if "feline" in query or "cat" in query: query_vector["cat_vibe"] = 0.9
        if "lodging" in query or "hotel" in query: query_vector["hotel_vibe"] = 0.9
        if "model-x" in query or "hardware" in query: query_vector["router_vibe"] = 0.9
        if "giraffe" in query or "safari" in query: query_vector["animal_vibe"] = 0.9

        scores = {}
        for doc_id, doc_vector in self.semantic_database.items():
            # Dot Product to calculate similarity score
            similarity = sum(query_vector[vibe] * doc_vector[vibe] for vibe in query_vector)
            scores[doc_id] = similarity
            
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [5]:
# ---------------------------------------------------------
# 4. THE MANAGER: HYBRID SEARCH FUSION (RRF Method)
# ---------------------------------------------------------
def hybrid_search(query, bm25_engine, vector_engine, k=1):
    # Step 1: Run both pipelines in parallel
    bm25_results = bm25_engine.retrieve(query)
    vector_results = vector_engine.retrieve(query)
    
    print(f"🤖 BM25 Robot List:   {bm25_results}")
    print(f"🧠 Vector Reader List: {vector_results}\n")
    
    # Step 2: Use Reciprocal Rank Fusion (RRF) to merge lists based strictly on ranking
    rrf_scores = {}
    
    # Process BM25 positions
    for rank, (doc_id, _) in enumerate(bm25_results, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k + rank))
        
    # Process Vector positions
    for rank, (doc_id, _) in enumerate(vector_results, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k + rank))
        
    # Step 3: Sort the merged dictionary to find the absolute winner
    final_ranking = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return final_ranking

In [8]:
# ---------------------------------------------------------
# 5. TEST DRIVING THE SYSTEM
# ---------------------------------------------------------
# Initialize engines
bm25 = BM25Retriever(documents)
vectors = SimulatedVectorRetriever()

# The tricky query containing: 
# "felines" & "lodging" (Synonyms for doc1) AND "model-X" (Exact keyword match for doc2)
user_query = "felines lodging model-X"

print(f"🔎 EXECUTING HYBRID SEARCH FOR: '{user_query}'")
print("-" * 60)

final_results = hybrid_search(user_query, bm25, vectors)

print("-" * 60)
print("🏆 FINAL HYBRID RANKING (Highest RRF score first; ties are possible):")
for rank, (doc_id, score) in enumerate(final_results, start=1):
    print(f"{rank}. {doc_id} (RRF Score: {score:.4f}) -> \"{documents[doc_id]}\"")

🔎 EXECUTING HYBRID SEARCH FOR: 'felines lodging model-X'
------------------------------------------------------------
🤖 BM25 Robot List:   [('doc2', 1.0917668525444708), ('doc1', 0.9952768197500332), ('doc3', 0.0)]
🧠 Vector Reader List: [('doc1', 1.665), ('doc2', 0.972), ('doc3', 0.09000000000000001)]

------------------------------------------------------------
🏆 FINAL HYBRID RANKING (Highest RRF score first; ties are possible):
1. doc2 (RRF Score: 0.8333) -> "A technical manual for router hardware model-X configurations."
2. doc1 (RRF Score: 0.8333) -> "The ultimate luxury hotel and resort resort resort for felines."
3. doc3 (RRF Score: 0.5000) -> "This short poem talks about a wild giraffe in its natural safari habitat."
